In [24]:
import pyspark
from pyspark.sql import SparkSession

In [25]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [26]:
filepath="D:/spark/datasets/indian_food.csv"

In [9]:
def createSparkSession():
    spark = SparkSession.builder.appName('indian-food').getOrCreate()
    return spark

In [5]:
def create_dataframe(spark, filepath):
    df = spark.read.csv(filepath, header=True, inferSchema=True)
    return df

In [13]:
spark = createSparkSession()

In [27]:
df = create_dataframe(spark, filepath)    

In [29]:
df = df.withColumn('id', F.monotonically_increasing_id())

In [30]:
df = df.select('id','name', 'flavor_profile', 'course', 'state')

In [34]:
from pyspark.sql.window import Window

In [37]:
window = (
    Window
    .orderBy('id')
    .rowsBetween(Window.unboundedPreceding, 0)
)

for c in df.columns:
    if c!='id':
        df = df.withColumn(c, F.last(
            F.when(F.col(c)=='-1', None)
            .otherwise(F.col(c)),
            ignorenulls = True
        ).over(window)
    )

In [92]:
north = [
    "Punjab",
    "Haryana",
    "Himachal Pradesh",
    "Uttarakhand",
    "Uttar Pradesh",
    "Rajasthan",
]

south = ["Tamil Nadu", "Kerala", "Karnataka", "Andhra Pradesh", "Telangana"]

east = [
    "Bihar",
    "Jharkhand",
    "Odisha",
    "West Bengal",
    "Assam",
    "Arunachal Pradesh",
    "Manipur",
    "Meghalaya",
    "Mizoram",
    "Nagaland",
    "Sikkim",
    "Tripura",
]

west = ["Maharashtra", "Gujarat", "Goa", "Madhya Pradesh", "Chhattisgarh"]

In [93]:
def mapregion(data):
    if data in north:
        return 'north'
    elif data in south:
        return 'south'
    elif data in east:
        return 'east'
    elif data in west:
        return 'west'

In [94]:
regionudf = udf(mapregion, StringType())
df = df.withColumn('region', regionudf(col('state')))
df.show()

+---+--------------+--------------+-------+-------------+------+
| id|          name|flavor_profile| course|        state|region|
+---+--------------+--------------+-------+-------------+------+
|  0|    Balu shahi|         sweet|dessert|  West Bengal|  east|
|  1|        Boondi|         sweet|dessert|    Rajasthan| north|
|  2|Gajar ka halwa|         sweet|dessert|       Punjab| north|
|  3|        Ghevar|         sweet|dessert|    Rajasthan| north|
|  4|   Gulab jamun|         sweet|dessert|  West Bengal|  east|
|  5|        Imarti|         sweet|dessert|  West Bengal|  east|
|  6|        Jalebi|         sweet|dessert|Uttar Pradesh| north|
|  7|    Kaju katli|         sweet|dessert|Uttar Pradesh| north|
|  8|      Kalakand|         sweet|dessert|  West Bengal|  east|
|  9|         Kheer|         sweet|dessert|  West Bengal|  east|
| 10|         Laddu|         sweet|dessert|  West Bengal|  east|
| 11|         Lassi|         sweet|dessert|       Punjab| north|
| 12|     Nankhatai|     

In [95]:
# 1. Find out how many unique dishes are present. 

In [96]:
unique_dishes = df.select('name').distinct().count()

In [97]:
unique_dishes

255

In [98]:
# 2. Which state has more dishes?

In [99]:
most_dishes_state = df.groupBy('state').count().orderBy('count', ascending=False).show(1)

+------+-----+
| state|count|
+------+-----+
|Punjab|   37|
+------+-----+
only showing top 1 row



In [100]:
# 3. How many dishes from state Karnataka?
karnataka_dishes_count = df.filter(df['state']=='Karnataka').count()
karnataka_dishes_count

7

In [104]:
# 4. List number of unique regions
unique_regions = df.select('region').distinct().count()
unique_regions

5

In [105]:
# 5. Count number of dishes from each region.
dishes_perregion = df.groupBy('region').count().show(truncate=False)


+------+-----+
|region|count|
+------+-----+
|NULL  |3    |
|west  |71   |
|east  |63   |
|north |58   |
|south |60   |
+------+-----+



In [106]:
# 6. List unique 'flavor_profile' and 'course' 
flavors = df.select('flavor_profile').distinct().count()
print(flavors)
courses = df.select('course').distinct().count()
print(courses)

4
4


In [107]:
# 7. Which state has more 'main course'?
main_courses = df.filter(df['course']=='main course').groupBy('state').count().orderBy(desc('count'))

In [108]:
main_courses.show(1)

+------+-----+
| state|count|
+------+-----+
|Punjab|   30|
+------+-----+
only showing top 1 row



In [109]:
# 8. Give the %of dishes from each region.
# state_counts = df.groupBy('state').count()
# w = Window.orderBy()
# res = state_counts.withColumn('percentage', F.col('count')/F.sum('counts').over(w))
# res.show()

total=df.count()
df.groupBy('region').count().withColumn('%',round(col('count')/total*100,2)).orderBy('%',ascending=False).show(100)

+------+-----+-----+
|region|count|    %|
+------+-----+-----+
|  west|   71|27.84|
|  east|   63|24.71|
| south|   60|23.53|
| north|   58|22.75|
|  NULL|    3| 1.18|
+------+-----+-----+



In [110]:
# 9. List the states which has more dishes from each region
df.select('region','state').groupBy('region').count().orderBy('count', ascending=False).show()

+------+-----+
|region|count|
+------+-----+
|  west|   71|
|  east|   63|
| south|   60|
| north|   58|
|  NULL|    3|
+------+-----+

